# CELEB-A with DDiT

In [4]:
import subprocess
from dotenv import load_dotenv
import os

load_dotenv()  # reads .env file in current directory

def set_modal_tokens():
    token_id     = os.environ.get("MODAL_TOKEN_ID")
    token_secret = os.environ.get("MODAL_TOKEN_SECRET")
    
    if not token_id or not token_secret:
        raise ValueError("MODAL_TOKEN_ID or MODAL_TOKEN_SECRET not found in .env")
    
    subprocess.run(
        ["modal", "token", "set",
         "--token-id", token_id,
         "--token-secret", token_secret],
        check=True
    )
    print("✅ Modal tokens configured")

set_modal_tokens()

Verifying token against https://api.modal.com
Token verified successfully!
⠋ Storing token
Token written to /root/.modal.toml in profile julio-2.
credentials.
✅ Modal tokens configured


In [ ]:
!modal run --detach celeba_dflow.py

In [1]:
%%writefile celeba_dflow.py
"""
DMAP DiT — CelebA 64×64  (Flow Matching),
══════════════════════════════════════════
- 64×64 RGB images → 256 tokens (patch_size=4, 4×4×3=48 dim)
- DMAP attention: logit = -½‖q_i - q_j‖² (Gaussian kernel)
- Flow matching: straight paths x_t = (1-t)·noise + t·x0
                 model learns velocity v = x0 - noise
- Unconditional (CelebA has no class labels)
- EMA of weights for sampling
- Warmup (10 ep) + plain cosine decay — no restarts
- Saves best EMA checkpoint separately
- Uploads to HuggingFace every 25 epochs

Key differences from celeba_dmap.py (DDPM):
  - No noise schedule (betas, abar etc.)
  - Continuous t ~ Uniform[0,1] not discrete t ~ {0,...,999}
  - Loss = MSE(model(x_t, t), x0 - noise)  not MSE(..., noise)
  - Sampler = Euler ODE not DDIM
  - Faster convergence, fewer NFE at inference

Modal B200 usage:
    modal run --detach celeba_dflow.py
"""

import modal

image = (
    modal.Image.debian_slim(python_version="3.12")
    .pip_install(
        "torch", "torchvision", "torchaudio",
        extra_index_url="https://download.pytorch.org/whl/cu128",
    )
    .pip_install(
        "huggingface_hub", "python-dotenv",
        "numpy", "matplotlib", "ninja",
        "datasets",
    )
)

app = modal.App("celeba-dflow-dit")

@app.function(
    image=image,
    gpu="B200",
    timeout=60 * 60 * 16,
    secrets=[modal.Secret.from_name("HF_TOKEN")],
)
def train():
    import os, math, time, json, signal, shutil, copy
    import numpy as np
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.amp import autocast
    from torch.utils.data import DataLoader, TensorDataset
    from torch.nn.attention.flex_attention import flex_attention
    from torchvision import transforms
    from huggingface_hub import HfApi, create_repo
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    import torch._dynamo
    torch._dynamo.config.recompile_limit = 64

    # ── config ────────────────────────────────────────────────────────────────
    MODE       = "dmap"
    DEVICE     = "cuda"
    DTYPE      = torch.bfloat16
    IMG_SIZE   = 64
    PATCH_SIZE = 4
    N_PATCHES  = (IMG_SIZE // PATCH_SIZE) ** 2   # 256
    PATCH_DIM  = PATCH_SIZE * PATCH_SIZE * 3     # 48
    D_MODEL    = 384
    N_HEADS    = 6
    N_LAYERS   = 12
    MLP_RATIO  = 4
    BATCH      = 256
    EPOCHS     = 400
    WARMUP_EP  = 10
    LR         = 3e-4
    LR_MIN     = 1e-6
    EMA_DECAY  = 0.9999
    BG         = "#0d0d0d"
    HF_REPO    = "/flt" #### <<<<<<<<<<<<<
    SM_OPTS    = {"BLOCK_M": 128, "BLOCK_N": 128}

    hf_token      = os.environ.get("HF_TOKEN")
    ckpt_path     = f"celeba_dflow_checkpoint.pt"
    best_ema_path = f"celeba_dflow_best_ema.pt"
    log_path      = f"celeba_dflow_results.log"

    print(f"\n{'='*60}")
    print(f"CelebA 64×64 DMAP-Flow DiT  |  {N_PATCHES} tokens  |  d={D_MODEL}")
    print(f"patch_size={PATCH_SIZE}  LR={LR}  warmup={WARMUP_EP}ep")
    print(f"Flow Matching (straight paths, Euler ODE sampler)")
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"{'='*60}\n")

    # ── HF upload helper ──────────────────────────────────────────────────────
    def hf_upload(api, hf_token, hf_repo, folder_path,
                  commit_message, timeout=120):
        def _timeout_handler(signum, frame):
            raise TimeoutError("HF upload timed out")
        signal.signal(signal.SIGALRM, _timeout_handler)
        for attempt in range(3):
            try:
                signal.alarm(timeout)
                create_repo(hf_repo, repo_type="dataset",
                            exist_ok=True, token=hf_token)
                api.upload_folder(folder_path=folder_path,
                                  repo_id=hf_repo, repo_type="dataset",
                                  commit_message=commit_message)
                signal.alarm(0)
                print(f"  → HF: {commit_message}", flush=True)
                return True
            except (TimeoutError, Exception) as e:
                signal.alarm(0)
                if attempt < 2:
                    print(f"  HF retry {attempt+1}: {e}", flush=True)
                    time.sleep(30*(attempt+1))
                else:
                    print("  HF upload failed — saved locally", flush=True)
                    return False

    # ── FlexAttention DMAP ────────────────────────────────────────────────────
    _flex = torch.compile(flex_attention, dynamic=True)

    def make_dmap_score_mod(q_norm2, k_norm2):
        def score_mod(score, b, h, qi, ki):
            return score - 0.5*(q_norm2[b,h,qi] + k_norm2[b,h,ki])
        return score_mod

    def dmap_flex_attn(q, k, v):
        qn = (q*q).sum(-1); kn = (k*k).sum(-1)
        return _flex(q, k, v,
                     score_mod=make_dmap_score_mod(qn, kn),
                     scale=1.0, kernel_options=SM_OPTS)

    # ── Flow matching ─────────────────────────────────────────────────────────
    def flow_sample(x0):
        """
        Sample a point on the straight path between noise and data.
        x_t = (1-t)·z + t·x0,   z ~ N(0,I),   t ~ U[0,1]
        target velocity v = x0 - z  (constant along path)
        Returns: x_t, v_target, t
        """
        B    = x0.shape[0]
        t    = torch.rand(B, device=x0.device)           # [B] ∈ [0,1]
        z    = torch.randn_like(x0, dtype=torch.float32) # noise
        t_   = t[:, None, None]                          # broadcast
        x_t  = (1 - t_) * z + t_ * x0.float()           # interpolate
        v    = x0.float() - z                            # target velocity
        return x_t, v, t

    @torch.no_grad()
    def euler_sample(model, n, steps=50):
        """
        Euler ODE integration from z~N(0,I) to x_data.
        dx/dt = v_θ(x_t, t),   t: 0→1
        """
        dt   = 1.0 / steps
        x    = torch.randn(n, N_PATCHES, PATCH_DIM, device=DEVICE)
        dev  = DEVICE.split(":")[0]
        model.eval()
        for i in range(steps):
            t_val = torch.full((n,), i * dt, device=DEVICE)
            with autocast(device_type=dev, dtype=DTYPE):
                v = model(x, t_val).float()
            x = x + v * dt
        return x

    @torch.no_grad()
    def heun_sample(model, n, steps=50):
        """
        Heun (2nd order) ODE integration — better quality, same NFE.
        """
        dt   = 1.0 / steps
        x    = torch.randn(n, N_PATCHES, PATCH_DIM, device=DEVICE)
        dev  = DEVICE.split(":")[0]
        model.eval()
        for i in range(steps):
            t0   = torch.full((n,), i * dt,       device=DEVICE)
            t1   = torch.full((n,), (i+1) * dt,   device=DEVICE)
            with autocast(device_type=dev, dtype=DTYPE):
                v0 = model(x, t0).float()
            x_e  = x + v0 * dt                    # Euler predictor
            with autocast(device_type=dev, dtype=DTYPE):
                v1 = model(x_e, t1).float()
            x    = x + 0.5 * (v0 + v1) * dt       # Heun corrector
        return x

    # ── Model blocks ──────────────────────────────────────────────────────────
    class TimestepEmbed(nn.Module):
        """
        Embeds continuous t ∈ [0,1] via sinusoidal features.
        Same as DDPM but t is now a float not an integer index.
        """
        def __init__(self, d):
            super().__init__(); self.d = d
            self.mlp = nn.Sequential(
                nn.Linear(d, d*4), nn.SiLU(), nn.Linear(d*4, d))
        def forward(self, t):
            # t: [B] float in [0,1]
            h = self.d // 2
            f = torch.exp(-math.log(10000) *
                          torch.arange(h, device=t.device) / (h - 1))
            a = t[:, None].float() * f[None] * 1000  # scale to match DDPM range
            return self.mlp(torch.cat([a.sin(), a.cos()], dim=-1))

    class AdaLNZero(nn.Module):
        def __init__(self, d):
            super().__init__()
            self.n1 = nn.LayerNorm(d, elementwise_affine=False)
            self.n2 = nn.LayerNorm(d, elementwise_affine=False)
            self.p  = nn.Sequential(nn.SiLU(), nn.Linear(d, 6*d))
            nn.init.zeros_(self.p[-1].weight)
            nn.init.zeros_(self.p[-1].bias)
        def forward(self, x, c):
            g1, b1, a1, g2, b2, a2 = self.p(c).chunk(6, dim=-1)
            x1 = (1 + g1[:, None]) * self.n1(x) + b1[:, None]
            x2 = (1 + g2[:, None]) * self.n2(x) + b2[:, None]
            return x1, x2, a1[:, None], a2[:, None]

    class DiTBlock(nn.Module):
        def __init__(self, d, nh, mlpr, mode):
            super().__init__()
            self.mode = mode; self.nh = nh; self.hd = d // nh
            self.adaln = AdaLNZero(d)
            self.qp = nn.Linear(d, d, bias=False)
            self.vp = nn.Linear(d, d, bias=False)
            if mode == "standard":
                self.kp = nn.Linear(d, d, bias=False)
            self.op  = nn.Linear(d, d)
            self.mlp = nn.Sequential(
                nn.Linear(d, d * mlpr), nn.GELU(), nn.Linear(d * mlpr, d))

        def attn(self, x):
            B, N, D = x.shape; H, Hd = self.nh, self.hd
            q = self.qp(x).view(B, N, H, Hd).transpose(1, 2)
            v = self.vp(x).view(B, N, H, Hd).transpose(1, 2)
            if self.mode == "standard":
                k   = self.kp(x).view(B, N, H, Hd).transpose(1, 2)
                out = F.scaled_dot_product_attention(q, k, v)
            elif self.mode == "sym":
                out = F.scaled_dot_product_attention(q, q, v)
            elif self.mode == "dmap":
                k = q
                if Hd >= 16:
                    out = dmap_flex_attn(q, k, v)
                else:
                    dots   = torch.matmul(q, k.transpose(-2, -1))
                    qn2    = (q * q).sum(-1, keepdim=True)
                    logits = dots - 0.5 * (qn2 + qn2.transpose(-2, -1))
                    out    = torch.matmul(F.softmax(logits, dim=-1), v)
            return out.transpose(1, 2).reshape(B, N, D)

        def forward(self, x, c):
            x1, x2, a1, a2 = self.adaln(x, c)
            x = x + a1 * self.op(self.attn(x1))
            x = x + a2 * self.mlp(x2)
            return x

    class DiT(nn.Module):
        def __init__(self, mode):
            super().__init__()
            self.mode = mode
            D = D_MODEL
            self.patch_embed = nn.Linear(PATCH_DIM, D)
            self.pos_embed   = nn.Parameter(
                torch.randn(1, N_PATCHES, D) * 0.02)
            self.t_embed     = TimestepEmbed(D)
            self.blocks      = nn.ModuleList([
                DiTBlock(D, N_HEADS, MLP_RATIO, mode)
                for _ in range(N_LAYERS)])
            self.final_norm  = nn.LayerNorm(D)
            self.final_proj  = nn.Linear(D, PATCH_DIM)
            nn.init.zeros_(self.final_proj.weight)
            nn.init.zeros_(self.final_proj.bias)

        def patchify(self, x):
            B, C, H, W = x.shape; P = PATCH_SIZE
            x = x.reshape(B, C, H//P, P, W//P, P)
            x = x.permute(0, 2, 4, 3, 5, 1)
            return x.reshape(B, N_PATCHES, PATCH_DIM)

        def unpatchify(self, x):
            B = x.shape[0]; P = PATCH_SIZE; G = IMG_SIZE // P
            x = x.reshape(B, G, G, P, P, 3)
            x = x.permute(0, 5, 1, 3, 2, 4)
            return x.reshape(B, 3, IMG_SIZE, IMG_SIZE)

        def forward(self, x, t):
            # t: [B] float in [0,1]
            x = self.patch_embed(x) + self.pos_embed
            c = self.t_embed(t)
            for block in self.blocks:
                x = block(x, c)
            return self.final_proj(self.final_norm(x))

    # ── EMA ───────────────────────────────────────────────────────────────────
    class EMA:
        def __init__(self, model, decay=0.9999):
            self.decay  = decay
            self.shadow = copy.deepcopy(model)
            self.shadow.eval()
            for p in self.shadow.parameters():
                p.requires_grad_(False)

        @torch.no_grad()
        def update(self, model):
            for s, m in zip(self.shadow.parameters(),
                            model.parameters()):
                s.data = self.decay * s.data + (1 - self.decay) * m.data

        def state_dict(self):        return self.shadow.state_dict()
        def load_state_dict(self, sd): self.shadow.load_state_dict(sd)

    # ── Dataset ───────────────────────────────────────────────────────────────
    print("Loading CelebA from HuggingFace...", flush=True)
    from datasets import load_dataset as hf_load
    from PIL import Image as PILImage

    hf_ds = hf_load("nielsr/CelebA-faces", split="train",
                    trust_remote_code=False)

    normalize = transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    resize    = transforms.Resize(
        (IMG_SIZE, IMG_SIZE),
        interpolation=transforms.InterpolationMode.BICUBIC)
    to_tensor = transforms.ToTensor()

    imgs_list = []
    for ex in hf_ds:
        img = ex.get("image", ex.get("img"))
        if not isinstance(img, PILImage.Image):
            img = PILImage.fromarray(img)
        imgs_list.append(normalize(to_tensor(resize(img.convert("RGB")))))

    imgs     = torch.stack(imgs_list)
    imgs_gpu = imgs.to(DEVICE, dtype=DTYPE)
    print(f"  Dataset on VRAM: {imgs_gpu.shape}  "
          f"{imgs_gpu.nbytes/1e9:.2f}GB", flush=True)

    loader = DataLoader(
        TensorDataset(imgs_gpu),
        batch_size=BATCH, shuffle=True,
        num_workers=0, pin_memory=False, drop_last=True)

    # ── Model + EMA + optimizer ───────────────────────────────────────────────
    model = DiT(MODE).to(DEVICE)
    ema   = EMA(model, decay=EMA_DECAY)
    n_p   = sum(p.numel() for p in model.parameters())
    print(f"Params: {n_p/1e6:.2f}M  tokens={N_PATCHES}  "
          f"d={D_MODEL}  H={N_HEADS}  L={N_LAYERS}", flush=True)

    optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

    def lr_lambda(epoch):
        if epoch < WARMUP_EP:
            return (epoch + 1) / WARMUP_EP
        progress = (epoch - WARMUP_EP) / max(1, EPOCHS - WARMUP_EP)
        cosine   = 0.5 * (1.0 + math.cos(math.pi * progress))
        return LR_MIN/LR + (1.0 - LR_MIN/LR) * cosine

    sched = torch.optim.lr_scheduler.LambdaLR(optim, lr_lambda)

    losses    = []
    best_loss = float('inf')
    t0        = time.time()
    api       = HfApi(token=hf_token)

    # ── Resume ────────────────────────────────────────────────────────────────
    start_epoch = 1
    if os.path.exists(ckpt_path):
        print(f"Resuming from {ckpt_path}", flush=True)
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model"])
        ema.load_state_dict(ckpt["ema"])
        optim.load_state_dict(ckpt["optim"])
        sched.load_state_dict(ckpt["sched"])
        losses      = ckpt["losses"]
        best_loss   = ckpt.get("best_loss", min(losses))
        start_epoch = ckpt["epoch"] + 1
        print(f"  resumed ep={start_epoch}  "
              f"last={losses[-1]:.5f}  best={best_loss:.5f}", flush=True)
    else:
        print("Starting from scratch", flush=True)

    # ── Training loop ─────────────────────────────────────────────────────────
    for epoch in range(start_epoch, EPOCHS + 1):
        model.train()
        tot = 0.0

        for (x0,) in loader:
            # ── flow matching forward ──────────────────────────────────────
            patches      = model.patchify(x0.float())
            x_t, v, t_f = flow_sample(patches)   # all float32

            with autocast(device_type="cuda", dtype=DTYPE):
                v_pred = model(x_t, t_f)
                loss   = F.mse_loss(v_pred.float(), v)

            optim.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()
            ema.update(model)
            tot += loss.item()

        sched.step()
        avg        = tot / len(loader)
        current_lr = optim.param_groups[0]['lr']
        losses.append(avg)

        # ── save best EMA ──────────────────────────────────────────────────
        if avg < best_loss:
            best_loss = avg
            torch.save({
                "epoch": epoch, "loss": avg,
                "ema":   ema.state_dict(),
                "meta":  {"mode": MODE, "type": "flow",
                           "d_model": D_MODEL, "n_heads": N_HEADS,
                           "n_layers": N_LAYERS, "n_patches": N_PATCHES,
                           "patch_dim": PATCH_DIM, "img_size": IMG_SIZE,
                           "patch_size": PATCH_SIZE},
            }, best_ema_path)
            print(f"  ✓ best EMA  ep={epoch}  loss={avg:.5f}", flush=True)

        # ── regular checkpoint ─────────────────────────────────────────────
        torch.save({
            "epoch": epoch, "model": model.state_dict(),
            "ema":   ema.state_dict(), "optim": optim.state_dict(),
            "sched": sched.state_dict(), "losses": losses,
            "best_loss": best_loss,
            "meta":  {"mode": MODE, "type": "flow",
                       "d_model": D_MODEL, "n_heads": N_HEADS,
                       "n_layers": N_LAYERS, "n_patches": N_PATCHES,
                       "patch_dim": PATCH_DIM, "img_size": IMG_SIZE,
                       "patch_size": PATCH_SIZE},
        }, ckpt_path)

        if epoch % 25 == 0:
            elapsed = (time.time() - t0) / 60
            print(f"ep {epoch:3d}/{EPOCHS}  loss={avg:.5f}  "
                  f"best={best_loss:.5f}  lr={current_lr:.2e}  "
                  f"t={elapsed:.1f}min", flush=True)

            # ── sample from EMA using both Euler and Heun ─────────────────
            ema.shadow.eval()
            for sampler_name, sampler_fn in [
                    ("euler", euler_sample),
                    ("heun",  heun_sample)]:
                with torch.no_grad():
                    samples = sampler_fn(ema.shadow, n=16, steps=50)
                    imgs_s  = ema.shadow.unpatchify(samples)
                    imgs_s  = ((imgs_s.clamp(-1,1)+1)/2).cpu().float().numpy()

                fig, axes = plt.subplots(4, 4, figsize=(8, 8))
                fig.patch.set_facecolor(BG)
                for idx, ax in enumerate(axes.flat):
                    ax.imshow(imgs_s[idx].transpose(1,2,0),
                              interpolation="nearest")
                    ax.axis("off")
                plt.suptitle(
                    f"[dflow/{sampler_name}/celeba64]  ep={epoch}  "
                    f"loss={avg:.5f}  best={best_loss:.5f}  "
                    f"EMA  steps=50",
                    color="white", fontsize=9)
                plt.tight_layout()
                sname = (f"celeba_dflow_{sampler_name}"
                         f"_ep{epoch:03d}_samples.png")
                plt.savefig(sname, dpi=130,
                            bbox_inches="tight", facecolor=BG)
                plt.close()

            # ── save log ──────────────────────────────────────────────────
            with open(log_path, "w") as f:
                json.dump({
                    "mode": MODE, "type": "flow",
                    "dataset": "celeba64", "epochs": EPOCHS,
                    "best_loss": best_loss,
                    "final_loss": losses[-1], "losses": losses,
                }, f, indent=2)

            # ── upload to HF ──────────────────────────────────────────────
            os.makedirs("hf_upload/logs",        exist_ok=True)
            os.makedirs("hf_upload/checkpoints", exist_ok=True)
            os.makedirs("hf_upload/samples",     exist_ok=True)
            shutil.copy(log_path,      f"hf_upload/logs/{log_path}")
            shutil.copy(ckpt_path,     f"hf_upload/checkpoints/{ckpt_path}")
            shutil.copy(best_ema_path, f"hf_upload/checkpoints/{best_ema_path}")
            for sampler_name in ["euler", "heun"]:
                sname = (f"celeba_dflow_{sampler_name}"
                         f"_ep{epoch:03d}_samples.png")
                if os.path.exists(sname):
                    shutil.copy(sname, f"hf_upload/samples/{sname}")
            hf_upload(api, hf_token, HF_REPO, "hf_upload",
                      f"celeba dflow ep{epoch}")

    print(f"\nDone.  final={losses[-1]:.5f}  best={best_loss:.5f}")
    return {"mode": MODE, "type": "flow",
            "final_loss": losses[-1],
            "best_loss": best_loss, "losses": losses}


@app.local_entrypoint()
def main():
    print("Launching CelebA 64×64 DMAP-Flow DiT on B200...")
    call = train.spawn()
    print(f"Job ID: {call.object_id}")
    result = call.get()
    print(f"Final: {result['final_loss']:.5f}  "
          f"Best: {result['best_loss']:.5f}")

Writing celeba_dflow.py
